In [ ]:
from collections.abc import Mapping
from pathlib import Path
import json
import sys
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torchvision import transforms
import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage, display

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / "src").exists() else NOTEBOOK_CWD.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from cody_jepa.data import (
    HealthGaitLoaderConfig,
    build_healthgait_datasets_from_config,
    build_healthgait_loaders_from_config,
    find_repo_root,
    healthgait_manifest_path,
    preview_manifest,
    summarize_healthgait_manifest,
    write_healthgait_metadata_summary,
)


CONFIG = {
    "seed": 0,

    "batch_size": 64,
    "num_epochs": 25,
    "steps": 1500,
    "lr": 1e-4,
    "ema_tau": 0.9995,
    
    "num_frames": 16,
    "img_size": 72,
    "patch_size": 24,
    "num_tokens": 144,
    "token_dim": 576,

    "embed_dim": 128,
    "hidden_dim": 512,
    "pred_dim": 256,
    "pred_depth": 2,
    "num_masked": 54,
    "block_size": 9,
    "num_heads": 4,
    "num_layers": 4,
}

In [ ]:
REPO_ROOT = find_repo_root()
MANIFEST_CSV = healthgait_manifest_path(REPO_ROOT)
LOADER_CONFIG = HealthGaitLoaderConfig(
    manifest_csv=MANIFEST_CSV,
    repo_root=REPO_ROOT,
    split="train",
    clip_length=CONFIG["num_frames"],
    image_size=(CONFIG["img_size"], CONFIG["img_size"]),
    batch_size=CONFIG["batch_size"],
    seed=CONFIG["seed"],
    num_workers=8,
    pin_memory=True
)

print(f"repo root: {REPO_ROOT}")
print(f"manifest:  {MANIFEST_CSV}")
print(f"exists:    {MANIFEST_CSV.exists()}")
print("loader config:")
print(json.dumps(LOADER_CONFIG.as_dict(), indent=2, sort_keys=True))

In [ ]:
preview_rows = preview_manifest(MANIFEST_CSV, n=3)

print("columns:", list(preview_rows[0].keys()) if preview_rows else [])
print("first row:")
display(preview_rows[0])

DIAGNOSTICS_DIR = REPO_ROOT / "data" / "healthgait" / "diagnostics"
manifest_summary = summarize_healthgait_manifest(
    MANIFEST_CSV,
    repo_root=REPO_ROOT,
    clip_length=LOADER_CONFIG.clip_length,
)
summary_paths = write_healthgait_metadata_summary(
    manifest_summary,
    DIAGNOSTICS_DIR,
    "healthgait_manifest_summary",
)
print(json.dumps(manifest_summary, indent=2, sort_keys=True))
print(f"saved: {summary_paths['json']}")
print(f"saved: {summary_paths['csv']}")

In [ ]:
diagnostic_epoch = 0

train_ds, val_ds = build_healthgait_datasets_from_config(LOADER_CONFIG)
train_ds.set_epoch(diagnostic_epoch)

print(f"train clips: {len(train_ds)}")
print(f"val clips:   {len(val_ds)}")

In [ ]:
train_loader, val_loader = build_healthgait_loaders_from_config(LOADER_CONFIG)

In [ ]:
batch = next(iter(train_loader))
video = batch["video"]
video.size()

In [ ]:
def multiblock_mask(cfg, batch_size=None, rng=None, device=None):
    B = cfg["batch_size"] if batch_size is None else int(batch_size)
    N = int(cfg["num_tokens"])
    block_size = int(cfg["block_size"])
    target = int(cfg["num_masked"])

    if B <= 0:
        raise ValueError(f"batch_size must be positive, got {B}")
    if block_size <= 0 or N % block_size != 0:
        raise ValueError("block_size must be positive and divide num_tokens")
    if target <= 0 or target >= N or target % block_size != 0:
        raise ValueError("num_masked must be positive, less than num_tokens, and divisible by block_size")

    rng = random.Random(cfg.get("seed", 0)) if rng is None else rng
    n_blocks = target // block_size
    block_ids = range(N // block_size)
    all_context = []
    all_targets = []

    for _ in range(B):
        sampled_blocks = rng.sample(block_ids, n_blocks)
        targets = torch.stack([
            torch.arange(block_id * block_size, (block_id + 1) * block_size)
            for block_id in sampled_blocks
        ])
        masked = torch.zeros(N, dtype=torch.bool)
        masked[targets.flatten()] = True
        all_context.append(torch.nonzero(~masked, as_tuple=True)[0])
        all_targets.append(targets)

    context_idx = torch.stack(all_context)
    target_blocks = torch.stack(all_targets)
    if device is not None:
        context_idx = context_idx.to(device)
        target_blocks = target_blocks.to(device)
    return context_idx, target_blocks

ctx_idx, target_blocks = multiblock_mask(CONFIG)

print("batch size: ", ctx_idx.size(0))
print("context size: ", ctx_idx.size(1))
print("target blocks: ", target_blocks.size(1))
print("target tokens: ", target_blocks.size(1) * target_blocks.size(2))

In [ ]:
ctx_idx[0]

In [ ]:
target_blocks[0]

In [ ]:
def token_to_frame_patch(token_idx, cfg):
    patches_per_side = cfg["img_size"] // cfg["patch_size"]
    patches_per_frame = patches_per_side * patches_per_side
    frame_idx = int(token_idx) // patches_per_frame
    patch_idx = int(token_idx) % patches_per_frame
    patch_row, patch_col = divmod(patch_idx, patches_per_side)
    return frame_idx, patch_row, patch_col

def visualize_mask(sample, context_idx, target_blocks, cfg=CONFIG):
    video = sample["video"].detach().cpu().clone()  # [T, C, H, W]
    if tuple(video.shape) != (cfg["num_frames"], 1, cfg["img_size"], cfg["img_size"]):
        raise ValueError(f"video shape {tuple(video.shape)} does not match CONFIG")

    original = video.repeat(1, 3, 1, 1)
    show = original * 0.25
    ps = cfg["patch_size"]
    colors = torch.tensor([
        [1.0, 0.15, 0.15],
        [0.15, 0.7, 1.0],
        [0.2, 1.0, 0.35],
        [1.0, 0.8, 0.1],
        [0.8, 0.3, 1.0],
        [1.0, 0.45, 0.1],
    ])

    for token_idx in context_idx:
        frame_idx, patch_row, patch_col = token_to_frame_patch(token_idx, cfg)
        y0 = patch_row * ps
        x0 = patch_col * ps
        show[frame_idx, :, y0:y0 + ps, x0:x0 + ps] = original[frame_idx, :, y0:y0 + ps, x0:x0 + ps]

    for block_idx, block in enumerate(target_blocks):
        color = colors[block_idx % len(colors)].view(3, 1, 1)
        for token_idx in block:
            frame_idx, patch_row, patch_col = token_to_frame_patch(token_idx, cfg)
            y0 = patch_row * ps
            x0 = patch_col * ps
            show[frame_idx, :, y0:y0 + ps, x0:x0 + ps] = color

    rows, cols = 4, 4
    _, axes = plt.subplots(rows, cols, figsize=(8, 8))
    for frame_idx, ax in enumerate(axes.flat):
        ax.imshow(show[frame_idx].permute(1, 2, 0).clamp(0, 1).numpy())
        ax.set_title(f"frame {frame_idx}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

visualize_mask(train_ds[0], ctx_idx[0], target_blocks[0])


In [ ]:
class AttentionBlock(nn.Module):

    def __init__(self, embed_dim, hidden_dim, num_heads, dropout=0.1):
        super().__init__()

        self.layer_norm_1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.layer_norm_2 = nn.LayerNorm(embed_dim)
        self.linear = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        residual = self.layer_norm_1(x)
        x = x + self.attn(residual, residual, residual, need_weights=False)[0]
        return x + self.linear(self.layer_norm_2(x))


In [ ]:
def img_to_patch(x, patch_size, flatten_channels=True):
    if x.ndim != 4:
        raise ValueError(f"expected images shaped [B, C, H, W], got {tuple(x.shape)}")

    B, C, H, W = x.shape
    if H % patch_size != 0 or W % patch_size != 0:
        raise ValueError(f"image size {(H, W)} must be divisible by patch_size={patch_size}")

    x = x.reshape(B, C, H // patch_size, patch_size, W // patch_size, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5)  # [B, H', W', C, p_H, p_W]
    x = x.flatten(1, 2)  # [B, H' * W', C, p_H, p_W]
    if flatten_channels:
        x = x.flatten(2, 4)  # [B, H' * W', C * p_H * p_W]
    return x

class VisionTransformer(nn.Module):

    def __init__(self, embed_dim, hidden_dim, token_dim, num_heads,
                 num_layers, patch_size, num_patches, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = int(num_patches)

        self.input_layer = nn.Linear(token_dim, embed_dim)
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)
        self.dropout = nn.Dropout(dropout)
        self.transformer = nn.Sequential(*[
            AttentionBlock(embed_dim, hidden_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.output_layer = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
        )

    @staticmethod
    def _batched_indices(indices, batch_size, num_tokens, device):
        indices = torch.as_tensor(indices, dtype=torch.long, device=device)
        if indices.ndim == 1:
            indices = indices.unsqueeze(0).expand(batch_size, -1)
        if indices.ndim != 2 or indices.size(0) != batch_size:
            raise ValueError(
                f"token indices must have shape [B, K] or [K], got {tuple(indices.shape)}"
            )
        if indices.numel() and (indices.min() < 0 or indices.max() >= num_tokens):
            raise IndexError(f"token indices must be in [0, {num_tokens})")
        return indices

    def forward(self, video, token_indices=None):
        if video.ndim == 4:
            video = video.unsqueeze(1)
        if video.ndim != 5:
            raise ValueError(
                f"expected video shaped [B, T, C, H, W], got {tuple(video.shape)}"
            )

        B, T, C, H, W = video.shape
        frame_patches = img_to_patch(
            video.reshape(B * T, C, H, W), self.patch_size
        )
        patches_per_frame = frame_patches.size(1)
        patches = frame_patches.reshape(B, T * patches_per_frame, -1)
        if patches.size(-1) != self.input_layer.in_features:
            raise ValueError(
                f"patches have width {patches.size(-1)} but token_dim is "
                f"{self.input_layer.in_features}"
            )
        N = patches.size(1)
        if N != self.num_patches:
            raise ValueError(
                f"encoder configured for {self.num_patches} tokens but video produced {N}"
            )

        x = self.input_layer(patches)
        pos = self.pos_embedding.expand(B, -1, -1)
        if token_indices is not None:
            token_indices = self._batched_indices(token_indices, B, N, video.device)
            gather_idx = token_indices.unsqueeze(-1).expand(-1, -1, x.size(-1))
            x = torch.gather(x, 1, gather_idx)
            pos = torch.gather(pos, 1, gather_idx)

        x = self.dropout(x + pos)
        x = self.transformer(x)
        return self.output_layer(x)  # [B, N, D] (or [B, K, D] when indexed)

In [ ]:
class Predictor(nn.Module):

    def __init__(self, dim, pred_dim, depth, n_heads, n_tokens, dropout=0.1):
        super().__init__()
        self.n_tokens = n_tokens
        self.embed = nn.Linear(dim, pred_dim)
        self.pos = nn.Parameter(torch.zeros(1, n_tokens, pred_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, pred_dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.blocks = nn.ModuleList([
            AttentionBlock(pred_dim, pred_dim * 4, n_heads, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(pred_dim)
        self.out = nn.Linear(pred_dim, dim)

    def _position_tokens(self, indices, batch_size):
        indices = VisionTransformer._batched_indices(
            indices, batch_size, self.n_tokens, self.pos.device
        )
        pos = self.pos.expand(batch_size, -1, -1)
        gather_idx = indices.unsqueeze(-1).expand(-1, -1, pos.size(-1))
        return torch.gather(pos, 1, gather_idx), indices

    def forward(self, ctx_tokens, ctx_idx, tgt_idx):
        if ctx_tokens.ndim != 3:
            raise ValueError(f"ctx_tokens must be [B, K, D], got {tuple(ctx_tokens.shape)}")

        B = ctx_tokens.size(0)
        ctx_pos, ctx_idx = self._position_tokens(ctx_idx, B)
        tgt_pos, tgt_idx = self._position_tokens(tgt_idx, B)
        if ctx_tokens.size(1) != ctx_idx.size(1):
            raise ValueError("ctx_tokens and ctx_idx must contain the same number of tokens")
        if tgt_idx.size(1) == 0:
            raise ValueError("tgt_idx must contain at least one target token")

        context = self.embed(ctx_tokens) + ctx_pos
        masked = self.mask_token.expand(B, tgt_idx.size(1), -1) + tgt_pos
        h = torch.cat([context, masked], dim=1)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h[:, -tgt_idx.size(1):, :])
        return self.out(h)  # [B, M, D]

In [ ]:
@torch.no_grad()
def ema_update(target, online, tau):
    if not 0.0 <= tau <= 1.0:
        raise ValueError(f"tau must be in [0, 1], got {tau}")

    target_params = dict(target.named_parameters())
    online_params = dict(online.named_parameters())
    if target_params.keys() != online_params.keys():
        raise ValueError("target and online encoders have different parameter structures")
    for name, target_param in target_params.items():
        target_param.mul_(tau).add_(online_params[name], alpha=1.0 - tau)

    target_buffers = dict(target.named_buffers())
    online_buffers = dict(online.named_buffers())
    if target_buffers.keys() != online_buffers.keys():
        raise ValueError("target and online encoders have different buffer structures")
    for name, target_buffer in target_buffers.items():
        target_buffer.copy_(online_buffers[name])

In [ ]:
def video_from_batch(batch, device):
    if not isinstance(batch, Mapping):
        raise TypeError(f"expected a DataLoader mapping, got {type(batch).__name__}")
    if "video" not in batch:
        raise KeyError("DataLoader batch is missing the 'video' field")

    video = batch["video"]
    if not isinstance(video, torch.Tensor):
        raise TypeError(f"batch['video'] must be a tensor, got {type(video).__name__}")
    if video.ndim != 5:
        raise ValueError(f"batch['video'] must be [B, T, C, H, W], got {tuple(video.shape)}")
    return video.to(device, non_blocking=True)


In [ ]:
def validate_jepa_config(cfg):
    positive_keys = (
        "num_epochs", "batch_size", "num_frames", "img_size", "patch_size", 
        "num_tokens", "token_dim", "embed_dim", "hidden_dim", "pred_dim", 
        "pred_depth", "num_masked", "block_size", "num_heads", "num_layers", "steps",
    )
    for key in positive_keys:
        if int(cfg[key]) <= 0:
            raise ValueError(f"cfg['{key}'] must be positive")
    if float(cfg["lr"]) <= 0.0:
        raise ValueError("cfg['lr'] must be positive")
    if cfg["img_size"] % cfg["patch_size"] != 0:
        raise ValueError("img_size must be divisible by patch")

    patches_per_frame = (cfg["img_size"] // cfg["patch_size"]) ** 2
    expected_tokens = cfg["num_frames"] * patches_per_frame
    if cfg["num_tokens"] != expected_tokens:
        raise ValueError(
            f"num_tokens={cfg['num_tokens']} but frames and patch geometry produce {expected_tokens}"
        )
    if cfg["embed_dim"] % cfg["num_heads"] != 0:
        raise ValueError("embed_dim must be divisible by num_heads")
    if cfg["pred_dim"] % cfg["num_heads"] != 0:
        raise ValueError("pred_dim must be divisible by num_heads")
    if not 0.0 <= cfg["ema_tau"] <= 1.0:
        raise ValueError("ema_tau must be in [0, 1]")
    multiblock_mask(cfg, batch_size=1)  # validates mask geometry


def gather_tokens(tokens, indices):
    indices = VisionTransformer._batched_indices(
        indices, tokens.size(0), tokens.size(1), tokens.device
    )
    gather_idx = indices.unsqueeze(-1).expand(-1, -1, tokens.size(-1))
    return torch.gather(tokens, 1, gather_idx)


def jepa_step(ctx_enc, tgt_enc, predictor, video, context_idx, target_blocks):
    target_idx = target_blocks.flatten(1)
    context_tokens = ctx_enc(video, context_idx)
    with torch.no_grad():
        target_tokens = gather_tokens(tgt_enc(video), target_idx)
    predicted_tokens = predictor(context_tokens, context_idx, target_idx)
    loss = F.smooth_l1_loss(predicted_tokens, target_tokens)
    cosine = F.cosine_similarity(predicted_tokens, target_tokens, dim=-1).mean()
    return loss, cosine


def train_jepa(cfg, train_loader, val_loader, device=None):
    validate_jepa_config(cfg)
    seed = int(cfg.get("seed", 0))
    torch.manual_seed(seed)

    if device is None:
        if torch.cuda.is_available():
            device = torch.device("cuda")
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
    else:
        device = torch.device(device)

    encoder_args = (
        cfg["embed_dim"], cfg["hidden_dim"], cfg["token_dim"],
        cfg["num_heads"], cfg["num_layers"], cfg["patch_size"], cfg["num_tokens"],
    )
    ctx_enc = VisionTransformer(*encoder_args).to(device)
    tgt_enc = VisionTransformer(*encoder_args).to(device)
    tgt_enc.load_state_dict(ctx_enc.state_dict())
    tgt_enc.requires_grad_(False)
    tgt_enc.eval()

    predictor = Predictor(
        cfg["embed_dim"], cfg["pred_dim"], cfg["pred_depth"],
        cfg["num_heads"], cfg["num_tokens"],
    ).to(device)
    optimizer = optim.Adam(
        [*ctx_enc.parameters(), *predictor.parameters()], lr=cfg["lr"]
    )

    train_mask_rng = random.Random(seed)
    max_steps = int(cfg["steps"])
    global_step = 0
    history = []

    for epoch in range(int(cfg["num_epochs"])):
        if global_step >= max_steps:
            break
        train_dataset = getattr(train_loader, "dataset", None)
        if hasattr(train_dataset, "set_epoch"):
            train_dataset.set_epoch(epoch)

        ctx_enc.train()
        predictor.train()
        tgt_enc.eval()
        train_loss_sum = 0.0
        train_cosine_sum = 0.0
        train_examples = 0

        for batch in train_loader:
            if global_step >= max_steps:
                break
            video = video_from_batch(batch, device)
            context_idx, target_blocks = multiblock_mask(
                cfg, batch_size=video.size(0), rng=train_mask_rng, device=device
            )
            loss, cosine = jepa_step(
                ctx_enc, tgt_enc, predictor, video, context_idx, target_blocks
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            ema_update(tgt_enc, ctx_enc, cfg["ema_tau"])

            batch_size = video.size(0)
            train_loss_sum += loss.detach().item() * batch_size
            train_cosine_sum += cosine.detach().item() * batch_size
            train_examples += batch_size
            global_step += 1

        if train_examples == 0:
            raise RuntimeError("train_loader produced no batches")

        ctx_enc.eval()
        predictor.eval()
        tgt_enc.eval()
        val_mask_rng = random.Random(seed + 1)
        val_loss_sum = 0.0
        val_cosine_sum = 0.0
        val_examples = 0
        with torch.no_grad():
            for batch in val_loader:
                video = video_from_batch(batch, device)
                context_idx, target_blocks = multiblock_mask(
                    cfg, batch_size=video.size(0), rng=val_mask_rng, device=device
                )
                loss, cosine = jepa_step(
                    ctx_enc, tgt_enc, predictor, video, context_idx, target_blocks
                )
                batch_size = video.size(0)
                val_loss_sum += loss.item() * batch_size
                val_cosine_sum += cosine.item() * batch_size
                val_examples += batch_size

        if val_examples == 0:
            raise RuntimeError("val_loader produced no batches")

        metrics = {
            "epoch": epoch + 1,
            "step": global_step,
            "train_loss": train_loss_sum / train_examples,
            "train_cosine": train_cosine_sum / train_examples,
            "val_loss": val_loss_sum / val_examples,
            "val_cosine": val_cosine_sum / val_examples,
        }
        history.append(metrics)
        print(
            f"epoch {metrics['epoch']:02d} | step {metrics['step']:04d} | "
            f"train loss {metrics['train_loss']:.4f}, cosine {metrics['train_cosine']:.4f} | "
            f"val loss {metrics['val_loss']:.4f}, cosine {metrics['val_cosine']:.4f}"
        )

    return {
        "context_encoder": ctx_enc,
        "target_encoder": tgt_enc,
        "predictor": predictor,
        "optimizer": optimizer,
        "history": history,
        "steps": global_step,
    }

In [ ]:
def smoke_test_training_loop():
    smoke_cfg = {
        "seed": 7, "num_epochs": 1, "batch_size": 2, "num_frames": 2,
        "img_size": 16, "patch_size": 8, "num_tokens": 8, "token_dim": 64,
        "embed_dim": 16, "hidden_dim": 32, "pred_dim": 24,
        "pred_depth": 1, "num_masked": 4, "block_size": 2,
        "num_heads": 4, "num_layers": 1, "steps": 1,
        "lr": 1e-3, "ema_tau": 0.9,
    }
    generator = torch.Generator().manual_seed(smoke_cfg["seed"])
    train_batches = [{
        "video": torch.rand(2, 2, 1, 16, 16, generator=generator),
        "subject_id": ["train-a", "train-b"],
    }]
    val_batches = [{
        "video": torch.rand(1, 2, 1, 16, 16, generator=generator),
        "subject_id": ["val-a"],
    }]

    result = train_jepa(smoke_cfg, train_batches, val_batches)
    metrics = result["history"][0]
    metric_values = torch.tensor([
        metrics["train_loss"], metrics["train_cosine"],
        metrics["val_loss"], metrics["val_cosine"],
    ])
    assert result["steps"] == 1
    assert torch.isfinite(metric_values).all()
    assert not result["target_encoder"].training
    assert not any(p.requires_grad for p in result["target_encoder"].parameters())
    return metrics


smoke_metrics = smoke_test_training_loop()
smoke_metrics

# Full run (intentionally opt-in):
# training = train_jepa(CONFIG, train_loader, val_loader)

In [ ]:
result = train_jepa(CONFIG, train_loader, val_loader)
result

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cody-jepa-haic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = OUTPUT_DIR / "01-cody-jepa.pt"

torch.save(
    {
        "config": CONFIG,
        "history": result["history"],
        "steps": result["steps"],
        "context_encoder": result["context_encoder"].state_dict(),
        "target_encoder": result["target_encoder"].state_dict(),
        "predictor": result["predictor"].state_dict(),
        "optimizer": result["optimizer"].state_dict(),
    },
    checkpoint_path
)

print(f"saved checkpoint: {checkpoint_path}")